# Stage 5 — SNOMED CT Entity Mapping

Map extracted clinical entities (symptoms, diagnoses mentioned, procedures, meds, labs)
from `patient_records/` to **offline SNOMED CT** (RF2 Snapshot under `data/SnomedCT_*`).

No cloud API — pure RF2 lexical match + fuzzy token/n-gram scoring.

**Input:** Stage 4 export  
**Output:** `data/stage_05_snomed_mapping/snomed_mappings.json`  
**Next:** `stage_06_snomed_ancestors.ipynb`

In [ ]:
import sys
from pathlib import Path

NB = Path.cwd()
if not (NB / "pipeline.py").exists():
    NB = NB.parent if (NB.parent / "notebooks" / "pipeline.py").exists() else NB
    if (NB / "notebooks" / "pipeline.py").exists():
        NB = NB / "notebooks"
sys.path.insert(0, str(NB))
REPO = NB.parent if NB.name == "notebooks" else NB

from pipeline import EXPORT_DIR, print_pipeline_banner
from snomed_ct import (
    build_snomed_index,
    collect_entities_from_patient_records,
    export_mappings_to_patient_folders,
    find_snomed_root,
    run_stage05_mapping,
    write_json,
)

print_pipeline_banner()
SNOMED_ROOT = find_snomed_root(REPO / "data")
STAGE_05_DIR = REPO / "data" / "stage_05_snomed_mapping"
STAGE_05_DIR.mkdir(parents=True, exist_ok=True)
print(f"SNOMED root : {SNOMED_ROOT}")
print(f"Export dir  : {EXPORT_DIR}")
print(f"Stage 5 out : {STAGE_05_DIR}")

In [ ]:
index = build_snomed_index(
    snomed_root=SNOMED_ROOT,
    cache_path=REPO / "data" / "snomed_index" / "snomed_index.pkl",
    force_rebuild=False,
)
print(f"Active concepts: {len(index.active_concepts):,}")

In [ ]:
entities = collect_entities_from_patient_records(EXPORT_DIR)
print(f"Entities: {len(entities)} | admissions: {len({(e['patient_id'], e['hadm_id']) for e in entities})}")

payload = run_stage05_mapping(entities, index)
print(f"Mapped: {payload['n_mapped']} | Unmapped: {payload['n_unmapped']}")

for row in payload["results"][:10]:
    s = row["snomed"]
    print(
        f"  [{row['kind']}] {row['term']!r}\n"
        f"    → {s.get('preferred_term')} | {s.get('concept_id')} | {s.get('match_method')} | score={s.get('score')}"
    )

In [ ]:
out = write_json(STAGE_05_DIR / "snomed_mappings.json", payload)
n = export_mappings_to_patient_folders(payload, EXPORT_DIR, "snomed_mapping.json")
print(f"Saved → {out}")
print(f"Per-admission files: {n}")
print("Next → stage_06_snomed_ancestors.ipynb")